# Trisha-v.3.0: Two-Stage Learning + Compound Scaling
3 model (EfficientFormer-L1, ConvNeXt-Tiny, CLIP ViT-B/32).
- **Stage 1:** Recyclable + Organic (160px)
- **Stage 2:** Organic + Electronic (256px)
- **Data:** backup → clean duplicate → keep leakage
- **Output:** `experiments/results/trisha_v3.0/`

In [3]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

%matplotlib inline

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
        .replace('experiments', '').rstrip(os.sep))

In [5]:
try:
    import open_clip
    print('open-clip-torch already installed')
except ImportError:
    print('Installing open-clip-torch...')

    import subprocess, sys
    try:
        subprocess.check_call(['uv', 'pip', 'install', 'open-clip-torch', '-q'])
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'open-clip-torch'])
    import open_clip
    print('Done')

Installing open-clip-torch...


CalledProcessError: Command '['d:\\Kuliah\\LOMBA\\Satria-Data\\big-data-big-trouble\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'open-clip-torch']' returned non-zero exit status 1.

In [4]:
from modules.utils.config import CLASS_LABELS, IMG_SIZE, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.models.clip_adapter import CLIPAdapter
from modules.training.loss import ClassBalancedLoss
from modules.training.train import fit
from modules.training.collator import MixUpCollator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = RESULTS / 'trisha_v3.0'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Output dir: {OUT_DIR}')
print(f'Project root: {PROJECT_ROOT}')

ModuleNotFoundError: No module named 'modules'

---
## 1. Setup Data: train_v3
Copy `train_backup/` → `train_v3/`, hapus 62 duplicate, rename 3 long-path.

In [ ]:
BACKUP_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_backup'
TRAIN_V3_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train_v3'

if not BACKUP_DIR.exists():
    raise FileNotFoundError('train_backup not found. Run data_quality.ipynb first.')

if TRAIN_V3_DIR.exists():
    print(f'{TRAIN_V3_DIR} already exists, skipping copy.')
else:
    print('Copying train_backup -> train_v3...')
    ret = os.system(f'robocopy "{BACKUP_DIR}" "{TRAIN_V3_DIR}" /E /COPY:DAT /R:2 /W:2 /NDL /NFL /NJH /NJS')
    print('Done')

In [ ]:
# Scan MD5 + remove duplicates (keep 1 per group)
hash_map = defaultdict(list)
def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p

for cls in sorted(os.listdir(TRAIN_V3_DIR)):
    cls_path = TRAIN_V3_DIR / cls
    if not cls_path.is_dir():
        continue
    for fpath in cls_path.iterdir():
        try:
            with open(_safe_path(fpath), 'rb') as f:
                h = hashlib.md5(f.read()).hexdigest()
            hash_map[h].append({'cls': cls, 'fname': fpath.name, 'path': fpath})
        except:
            pass

dup_groups = {h: files for h, files in hash_map.items() if len(files) > 1}
print(f'Duplicate groups: {len(dup_groups)}')

removed = 0
for h, files in dup_groups.items():
    for f in files[1:]:
        os.remove(f['path'])
        removed += 1
print(f'Removed {removed} duplicate files.')

In [ ]:
# Rename 3 long-path files
long_names = [
    ("1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-representasi-visual-yang-luar-biasa-dari-masalah-limbah-elektronik-yang-berkembang-dan-kebutuhan-akan-solusi-daur-ulang-berkelanjutan_73523-11403.jpg", "E_001.jpg"),
    ("12.pile-discarded-motherboards-cpus-cables-disc-drives-hijacked-hardware-heap-concept-electronic-waste-tech-recycling-hardware-reuse-sustainable-technology-environmental-impact_864588-263287.jpg", "E_012.jpg"),
    ("64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat-seperti-perangkat-elektronik-yang-mengeluarkan-bahan-kimia-beracun-ke-dalam-tanah-dan-saluran-air-di-lokasi-pembuangan-ilegal_997534-43151.jpg", "E_064.jpg"),
]
for old_name, new_name in long_names:
    old = TRAIN_V3_DIR / '1_Electronic' / old_name
    new = old.with_name(new_name)
    try:
        os.rename(_safe_path(old), _safe_path(new))
        print(f'{old_name[:40]}... -> {new_name}')
    except FileNotFoundError:
        pass
    except FileExistsError:
        pass
print('Rename done.')

---
## 2. Load & Split Data (train_v3)

In [ ]:
df = load_train('data/raw/train_v3')
print(f'Total: {len(df)}')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

---
## 3. Dataset Class + Transforms

In [ ]:
_LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}

def _open_image(path):
    safe = _safe_path(path)
    with open(safe, 'rb') as f:
        return Image.open(BytesIO(f.read())).convert('RGB')

class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = (PROJECT_ROOT / df['path']).values
        self.labels = df['label'].map(_LABEL_TO_IDX).values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = _open_image(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def make_transform(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.3, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
            transforms.RandomErasing(p=0.3),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

def make_loader(df, img_size, batch_size=32, shuffle=True, use_mixup=False):
    ds = TrashDataset(df, transform=make_transform(img_size, is_train=shuffle))
    collator = MixUpCollator(alpha=0.2, num_classes=3) if (use_mixup and shuffle) else None
    sampler = None
    if shuffle:
        labels = df['label'].map(_LABEL_TO_IDX).values
        counts = torch.bincount(torch.tensor(labels))
        weights = 1.0 / counts[labels].float()
        sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        ds, batch_size=batch_size,
        sampler=sampler, shuffle=False if sampler else shuffle,
        num_workers=4, pin_memory=torch.cuda.is_available(),
        collate_fn=collator,
    )

---
## 4. Dataloaders
Stage 1: Recyclable+Organic (160px)
Stage 2: Organic+Electronic (256px)
Final eval: full 3 kelas (224px)

In [ ]:
BATCH_SIZE = 32

# Stage 1: Recyclable + Organic
mask_s1 = train_df['label'].isin(['0_Recyclable', '2_Organic'])
train_s1 = train_df[mask_s1].reset_index(drop=True)
val_s1 = val_df[val_df['label'].isin(['0_Recyclable', '2_Organic'])].reset_index(drop=True)
train_loader_s1 = make_loader(train_s1, img_size=160, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s1 = make_loader(val_s1, img_size=160, shuffle=False)
print(f'Stage 1 train: {len(train_s1)}, val: {len(val_s1)}')

# Stage 2: Organic + Electronic
mask_s2 = train_df['label'].isin(['1_Electronic', '2_Organic'])
train_s2 = train_df[mask_s2].reset_index(drop=True)
val_s2 = val_df[val_df['label'].isin(['1_Electronic', '2_Organic'])].reset_index(drop=True)
train_loader_s2 = make_loader(train_s2, img_size=256, batch_size=BATCH_SIZE, use_mixup=True)
val_loader_s2 = make_loader(val_s2, img_size=256, shuffle=False)
print(f'Stage 2 train: {len(train_s2)}, val: {len(val_s2)}')

# Final full evaluation
train_loader_full = make_loader(train_df, img_size=224, use_mixup=False)
val_loader_full = make_loader(val_df, img_size=224, shuffle=False)
print(f'Full train: {len(train_df)}, val: {len(val_df)}')

---
## 5. Helper: move .pt to output dir

In [ ]:
def move_to_outdir(name):
    for ext in ['.pt', '.json']:
        src = RESULTS / f'{name}{ext}'
        if src.exists():
            shutil.move(str(src), str(OUT_DIR / f'{name}{ext}'))

def eval_model(model, loader):
    model.to(DEVICE).eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc='Eval', leave=False):
            out = model(x.to(DEVICE))
            preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(labels, preds, average='macro')
    f1_pc = f1_score(labels, preds, average=None, labels=list(range(3))).tolist()
    prec_pc = precision_score(labels, preds, average=None, labels=list(range(3))).tolist()
    rec_pc = recall_score(labels, preds, average=None, labels=list(range(3))).tolist()
    return {'f1_macro': f1, 'f1_per_class': f1_pc, 'precision': prec_pc, 'recall': rec_pc, 'preds': preds, 'labels': labels}

---
## 6. TRAIN: EfficientFormer-L1

In [ ]:
MODEL = 'efficientformer_l1'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 160px
print(f'\nStage 1: Recyclable + Organic @ 160px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 256px
print(f'\nStage 2: Organic + Electronic @ 256px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 7. TRAIN: ConvNeXt-Tiny

In [ ]:
MODEL = 'convnext_tiny'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 160px
print(f'\nStage 1: Recyclable + Organic @ 160px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 256px
print(f'\nStage 2: Organic + Electronic @ 256px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 8. TRAIN: CLIP ViT-B/32 (VLM)

In [ ]:
# Wrap CLIPAdapter to match fit() interface
class CLIPTrainable(CLIPAdapter):
    @property
    def classifier(self):
        return self.visual_projection
    @property
    def encoder(self):
        return self.clip.visual
    def freeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip.visual.parameters():
            p.requires_grad = True

MODEL = 'clip_vit_b32'
print(f'=== {MODEL} ===')

model = CLIPTrainable(clip_variant='ViT-B/32', num_classes=3, device=DEVICE)
clip_params = sum(p.numel() for p in model.clip.visual.parameters() if p.requires_grad)
proj_params = sum(p.numel() for p in model.visual_projection.parameters())
print(f'CLIP encoder: {clip_params:,}, Projection: {proj_params:,}')

# Zero-shot benchmark first
model.to(DEVICE).eval()
prompts = ['a photo of recyclable waste', 'a photo of electronic waste', 'a photo of organic waste']
zs_preds, zs_labels = [], []
for x, y in tqdm(val_loader_full, desc='Zero-shot', leave=False):
    logits = model.zero_shot_classify(x.to(DEVICE), prompts)
    zs_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    zs_labels.extend(y.numpy().tolist())
from sklearn.metrics import f1_score
print(f'Zero-shot F1 macro: {f1_score(zs_labels, zs_preds, average="macro"):.4f}')

# Stage 1: Recyclable + Organic, 160px (fine-tune projection only)
print(f'\nStage 1: Recyclable + Organic @ 160px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=10, epochs_finetune=15,
    lr_head=1e-3, lr_finetune=1e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 256px
print(f'\nStage 2: Organic + Electronic @ 256px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=4, epochs_head=6, epochs_finetune=10,
    lr_head=5e-4, lr_finetune=5e-6, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 8.1 TRAIN: Swin-Tiny

In [ ]:
MODEL = 'swin_tiny_patch4_window7_224'
print(f'=== {MODEL} ===')

model = TrashClassifier(encoder_name=MODEL, num_classes=3)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Stage 1: Recyclable + Organic, 160px
print(f'\nStage 1: Recyclable + Organic @ 160px')
fit(model, train_loader_s1, val_loader_s1,
    name=f'{MODEL}_s1', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=8, epochs_finetune=20,
    lr_head=5e-4, lr_finetune=5e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s1')

# Stage 2: Organic + Electronic, 256px
print(f'\nStage 2: Organic + Electronic @ 256px')
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s1.pt', map_location='cpu'))
fit(model, train_loader_s2, val_loader_s2,
    name=f'{MODEL}_s2', encoder_name=MODEL,
    accumulation_steps=2, epochs_head=6, epochs_finetune=15,
    lr_head=3e-4, lr_finetune=2e-5, patience=5,
    criterion=nn.CrossEntropyLoss(), device=DEVICE)
move_to_outdir(f'{MODEL}_s2')

# Final eval + save
model.load_state_dict(torch.load(OUT_DIR / f'{MODEL}_s2.pt', map_location=DEVICE))
res = eval_model(model, val_loader_full)
print(f'F1 macro: {res["f1_macro"]:.4f}')
print(classification_report(res['labels'], res['preds'], target_names=CLASS_LABELS, digits=4))
torch.save(model.state_dict(), OUT_DIR / f'{MODEL}.pt')
with open(OUT_DIR / f'{MODEL}.json', 'w') as f:
    json.dump({k: v for k, v in res.items() if k not in ['preds', 'labels']}, f, indent=2)
print(f'Saved: {OUT_DIR / MODEL}.pt/.json')

---
## 9. Summary Table

In [ ]:
models = ['efficientformer_l1', 'convnext_tiny', 'swin_tiny_patch4_window7_224', 'clip_vit_b32']
rows = []
for m in models:
    path = OUT_DIR / f'{m}.json'
    if path.exists():
        d = json.load(open(path))
        rows.append({
            'model': m,
            'params': f'{sum(p.numel() for p in TrashClassifier(encoder_name=m, num_classes=3).parameters()):,}' if m != 'clip_vit_b32' else '150M+',
            'f1_macro': d['f1_macro'],
            'f1_0_recyclable': d['f1_per_class'][0],
            'f1_1_electronic': d['f1_per_class'][1],
            'f1_2_organic': d['f1_per_class'][2],
            'precision': d['precision'],
            'recall': d['recall'],
        })

df_result = pd.DataFrame(rows).sort_values('f1_macro', ascending=False)
print(df_result.to_string(index=False))

df_result.to_csv(OUT_DIR / 'summary.csv', index=False)
print(f'\nSaved: {OUT_DIR / "summary.csv"}')